In [1]:
"""
Custom CNN - Binary Image Classification (Cats vs Dogs)
Beginner-friendly, well-commented training script.
"""

import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

# ------------------------------
# STEP 0: SETTINGS
# ------------------------------
NUM_CLASSES = 2          # binary: cats vs dogs
IMG_SIZE = (128, 128)
BATCH_SIZE = 16
EPOCHS = 20

TRAIN_DIR = "extracted/train"   # contains cats/ and dogs/
TEST_DIR  = "extracted/test"    # contains cats/ and dogs/

# ------------------------------
# STEP 1: LOAD DATA
# ------------------------------
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

class_names = train_ds.class_names
print("Classes found:", class_names)   # ['cats', 'dogs']

# ------------------------------
# STEP 2: PREPROCESSING
# ------------------------------
normalization_layer = layers.Rescaling(1./255)

data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y))
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds  = test_ds.map(lambda x, y: (normalization_layer(x), y))

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds  = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

# ------------------------------
# STEP 3: BUILD THE CNN
# ------------------------------
model = models.Sequential([
    layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),

    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D(2, 2),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(1, activation='sigmoid')   # binary output
])

# ------------------------------
# STEP 4: COMPILE
# ------------------------------
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ------------------------------
# STEP 5: TRAIN
# ------------------------------
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS
)

# ------------------------------
# STEP 6: EVALUATE ON TEST SET
# ------------------------------
test_loss, test_acc = model.evaluate(test_ds)
print(f"\nTest Accuracy: {test_acc*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

# ------------------------------
# STEP 7: PLOT ACCURACY & LOSS
# ------------------------------
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.legend()
plt.title('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Loss')

plt.tight_layout()
plt.savefig("training_curves.png", dpi=120)
print("Saved training_curves.png")

# ------------------------------
# STEP 8: SAVE MODEL
# ------------------------------
model.save("custom_cnn_catsdogs.keras")
print("Model saved as custom_cnn_catsdogs.keras")

FileNotFoundError: [Errno 2] No such file or directory: 'extracted/train'

The error `FileNotFoundError: [Errno 2] No such file or directory: 'extracted/train'` indicates that the script could not find the directory where the training images are supposed to be located.

To resolve this, you need to ensure that your dataset is correctly organized and accessible in your Colab environment. For the 'Cats vs Dogs' dataset, this typically means:

1.  **Download the dataset**: Often from a source like Kaggle.
2.  **Upload or Mount**: If downloaded locally, upload it to Colab, or mount your Google Drive if the data is there.
3.  **Extract the archive**: The dataset usually comes as a `.zip` file (e.g., `train.zip`, `test1.zip`). You'll need to extract these into the `extracted/train` and `extracted/test` directories as expected by the code.

Here's an example of how you might download and extract the dataset if it were available via a direct link or uploaded to your Colab session (assuming `train.zip` and `test.zip` are in the current working directory or accessible):


In [ ]:
import os
import shutil
import zipfile

# Define the target directories
BASE_DIR = 'extracted'
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
TEST_DIR = os.path.join(BASE_DIR, 'test') # Note: Kaggle's test data is often in 'test1'

# Create the base directory if it doesn't exist
os.makedirs(BASE_DIR, exist_ok=True)

# Path to the provided archive
ARCHIVE_PATH = '/content/archive.zip'

print(f"Processing {ARCHIVE_PATH}...")

# Step 1: Extract the main archive.zip
# This often contains train.zip and test1.zip for the Cats vs Dogs dataset.
if os.path.exists(ARCHIVE_PATH):
    with zipfile.ZipFile(ARCHIVE_PATH, 'r') as zip_ref:
        print(f"Extracting {ARCHIVE_PATH} to current directory.")
        zip_ref.extractall('.') # Extract contents to the current directory
else:
    print(f"Error: {ARCHIVE_PATH} not found. Please ensure it is uploaded.")
    # Exit or handle error as appropriate

# Step 2: Extract train.zip
if os.path.exists('train.zip'):
    os.makedirs(TRAIN_DIR, exist_ok=True)
    with zipfile.ZipFile('train.zip', 'r') as zip_ref:
        print(f"Extracting train.zip to {TRAIN_DIR}")
        zip_ref.extractall(TRAIN_DIR)
    os.remove('train.zip') # Clean up the intermediate zip file
else:
    print("train.zip not found after extracting archive.zip.")

# Step 3: Extract test1.zip (common for Kaggle Cats vs Dogs test data)
# And rename it to 'test' as expected by the model code
if os.path.exists('test1.zip'):
    # Extract to a temporary location first, then move contents
    temp_test_dir = os.path.join(BASE_DIR, 'test1_temp')
    os.makedirs(temp_test_dir, exist_ok=True)
    with zipfile.ZipFile('test1.zip', 'r') as zip_ref:
        print(f"Extracting test1.zip to {temp_test_dir}")
        zip_ref.extractall(temp_test_dir)
    os.remove('test1.zip') # Clean up the intermediate zip file

    # Assuming test1.zip extracts its content directly into 'temp_test_dir'
    # The actual test data (images) might be inside a subfolder like 'test1'
    # We need to find the actual images directory inside 'test1_temp'
    # The dataset usually has the test images directly under test1.zip's root

    # Move contents from temp_test_dir to TEST_DIR ('extracted/test')
    os.makedirs(TEST_DIR, exist_ok=True)
    for item in os.listdir(temp_test_dir):
        shutil.move(os.path.join(temp_test_dir, item), TEST_DIR)
    shutil.rmtree(temp_test_dir) # Remove the temporary directory
    print(f"Moved test data to {TEST_DIR}")

else:
    print("test1.zip not found after extracting archive.zip.")


print("Data extraction attempt complete. Please check the 'extracted' directory structure.")

# Verify the directory structure
print("\nVerifying directory structure:")
for root, dirs, files in os.walk(BASE_DIR):
    level = root.replace(BASE_DIR, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        if not f.startswith('.'): # Ignore hidden files like ._MACOSX
            print(f'{subindent}{f}')


After running the above cell and ensuring your `extracted/train` and `extracted/test` directories are populated correctly, you can re-run the original code cell to load the dataset.